In [7]:
#!/usr/bin/env python3
"""
Standalone end-to-end test — no Celery/Redis needed.
Trains and predicts directly using ModelTrainer and ModelPredictor.

Run: python test_ml_pipeline.py
"""

import os
import sys
import json
from pathlib import Path

os.environ["MODEL_DIR"] = "/tmp/test_models"
os.makedirs("/tmp/test_models", exist_ok=True)

# Allow imports from project root
try:
    project_root = Path(__file__).resolve().parent
except NameError:
    project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

from ml.trainer import ModelTrainer, DATASETS, MODELS
from ml.predictor import ModelPredictor

SEP = "─" * 60

In [8]:
def print_section(title: str):
    print(f"\n{SEP}")
    print(f"  {title}")
    print(SEP)


def test_training(dataset: str, model_type: str, job_id: str) -> dict:
    print_section(f"TRAINING  dataset={dataset}  model={model_type}")

    config = {
        "dataset": dataset,
        "model_type": model_type,
        "test_size": 0.2,
        "cv_folds": 5,
        "scale": True,
    }

    def on_epoch(epoch, total, metrics):
        bar = "█" * epoch + "░" * (total - epoch)
        print(f"  [{bar}] step {epoch}/{total}  {metrics}")

    trainer = ModelTrainer(config)
    trainer.load_data()
    result = trainer.train(epoch_callback=on_epoch)
    model_path = trainer.save(job_id)

    m = result["metrics"]
    print(f"\n  ✅ test_accuracy  : {m['test_accuracy']}")
    print(f"  ✅ f1_weighted    : {m['f1_score_weighted']}")
    print(f"  ✅ cv             : {m['cv_mean']} ± {m['cv_std']}")
    print(f"  ✅ saved to       : {model_path}")
    return result

In [9]:
def test_prediction(job_id: str, dataset: str):
    print_section(f"PREDICTION  model_id={job_id}")

    predictor = ModelPredictor.load(job_id)

    # ── Single prediction (named features) ────────────────────────────────────
    sample_named = {name: 5.0 for name in predictor.feature_names}
    result = predictor.predict(sample_named)
    print(f"\n  Named input  → label: {result['label']!r}  confidence: {result.get('confidence')}")

    # ── Single prediction (feature list) ──────────────────────────────────────
    sample_list = [5.1, 3.5, 1.4, 0.2] if dataset == "iris" else [0.5] * predictor.dataset_meta["n_features"]
    result2 = predictor.predict({"features": sample_list})
    print(f"  List input   → label: {result2['label']!r}  confidence: {result2.get('confidence')}")

    # ── Batch prediction ───────────────────────────────────────────────────────
    batch = [{"features": [v + i * 0.1 for v in sample_list]} for i in range(5)]
    batch_results = predictor.predict_batch(batch)
    labels = [r["label"] for r in batch_results]
    print(f"  Batch (5x)   → labels: {labels}")

    # ── Explain ────────────────────────────────────────────────────────────────
    explanation = predictor.explain({"features": sample_list})
    print(f"\n  Explanation for input {sample_list}:")
    print(f"    Predicted  : {explanation['prediction']['label']!r}")
    if explanation["feature_importances"]:
        print("    Top features:")
        for feat, imp in list(explanation["feature_importances"].items())[:3]:
            bar = "▓" * int(imp * 30)
            print(f"      {feat:<35} {bar} {imp:.4f}")
    else:
        print("    (feature importances not available for this model type)")

    # ── Model info ─────────────────────────────────────────────────────────────
    info = predictor.model_info()
    print(f"\n  Model info: {json.dumps(info, indent=4)}")

In [10]:
def run_all():
    print("\n🚀  ML Pipeline End-to-End Test\n")

    combos = [
        ("iris",           "random_forest",       "test-iris-rf"),
        ("wine",           "gradient_boosting",   "test-wine-gb"),
        ("breast_cancer",  "logistic_regression", "test-cancer-lr"),
        ("iris",           "svm",                 "test-iris-svm"),
    ]

    results = {}
    for dataset, model_type, job_id in combos:
        train_result = test_training(dataset, model_type, job_id)
        test_prediction(job_id, dataset)
        results[job_id] = train_result["metrics"]["test_accuracy"]

    print_section("SUMMARY")
    print(f"\n  {'Job ID':<30} {'Accuracy':>10}")
    print(f"  {'─'*30} {'─'*10}")
    for jid, acc in results.items():
        stars = "★" * int(acc * 5)
        print(f"  {jid:<30} {acc:>10.4f}  {stars}")
    print()

In [11]:
if __name__ == "__main__":
    run_all()


🚀  ML Pipeline End-to-End Test


────────────────────────────────────────────────────────────
  TRAINING  dataset=iris  model=random_forest
────────────────────────────────────────────────────────────
  [█░░░░░░░░░] step 1/10  {'step': 1, 'trees': 10, 'train_accuracy': 1.0}
  [██░░░░░░░░] step 2/10  {'step': 2, 'trees': 20, 'train_accuracy': 1.0}
  [███░░░░░░░] step 3/10  {'step': 3, 'trees': 30, 'train_accuracy': 1.0}
  [████░░░░░░] step 4/10  {'step': 4, 'trees': 40, 'train_accuracy': 1.0}
  [█████░░░░░] step 5/10  {'step': 5, 'trees': 50, 'train_accuracy': 1.0}
  [██████░░░░] step 6/10  {'step': 6, 'trees': 60, 'train_accuracy': 1.0}
  [███████░░░] step 7/10  {'step': 7, 'trees': 70, 'train_accuracy': 1.0}
  [████████░░] step 8/10  {'step': 8, 'trees': 80, 'train_accuracy': 1.0}
  [█████████░] step 9/10  {'step': 9, 'trees': 90, 'train_accuracy': 1.0}
  [██████████] step 10/10  {'step': 10, 'trees': 100, 'train_accuracy': 1.0}
  [██████████] step 10/10  {'stage': 'done'}

  ✅ test_